In [0]:
# ═══════════════════════════════════════════════════════════
# NOTEBOOK 3: 5 REGRESSION ALGORITHMS + MLFLOW
# Regression example uses: LinearRegression,
# RandomForest, GBT, XGBoost(GPU), ElasticNet
# ═══════════════════════════════════════════════════════════

import time, mlflow, mlflow.spark
from pyspark.ml.regression import (LinearRegression, DecisionTreeRegressor,
    RandomForestRegressor, GBTRegressor)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

train_p = spark.read.parquet("/Volumes/workspace/default/fema_claims_project/test_prepared/")
val_p = spark.read.parquet("/Volumes/workspace/default/fema_claims_project/val_prepared/")
# Caching/persisting is not supported in serverless/Spark Connect, so we skip it.
# train_p.persist(); val_p.persist()

label = "amountPaidOnBuildingClaim"
evaluator = RegressionEvaluator(labelCol=label, metricName="rmse")
r2_eval = RegressionEvaluator(labelCol=label, metricName="r2")

# ── Define 5 regression models(Linear, RF, GBT, DecisionTree, ElasticNet) ──
models = {
    "LinearRegression": LinearRegression(featuresCol="features", labelCol=label, maxIter=100),
    "DecisionTree": DecisionTreeRegressor(featuresCol="features", labelCol=label, maxDepth=10),
    "RandomForest": RandomForestRegressor(featuresCol="features", labelCol=label, numTrees=100, maxDepth=10),
    "GBT": GBTRegressor(featuresCol="features", labelCol=label, maxIter=100, maxDepth=8),
    "ElasticNet": LinearRegression(featuresCol="features", labelCol=label, elasticNetParam=0.5, regParam=0.01),
}

# MLflow requires a Unity Catalog volume path for dfs_tmpdir in shared/serverless clusters.
mlflow_tmp_dir = "/Volumes/workspace/default/fema_claims_project/mlflow_tmp/"

# ── Train & Log ALL Models (or EACH model: train → evaluate → log to MLflow → store results) ──
reg_results = []
for name, model in models.items():
    with mlflow.start_run(run_name=f"REG_{name}"):
        t0 = time.time()
        fitted = model.fit(train_p)
        duration = time.time() - t0
        preds = fitted.transform(val_p)
        rmse = evaluator.evaluate(preds)
        r2 = r2_eval.evaluate(preds)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.log_metric("training_time_s", duration)
        # Pass dfs_tmpdir argument to log_model to specify the Unity Catalog volume path
        mlflow.spark.log_model(fitted, name, dfs_tmpdir=mlflow_tmp_dir)
        reg_results.append((name, rmse, r2, duration))
        print(f"✅ {name:20s} → RMSE: ${rmse:,.0f} | R²: {r2:.4f} | {duration:.1f}s")

# Show comparison table
reg_df = spark.createDataFrame(reg_results, ["model","rmse","r2","time_s"])
display(reg_df)

2026/03/02 21:34:25 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/02 21:34:29 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-fe02ac87-cb7b-4860-bcc0-26/tmpfbuiwbjg/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/02 21:34:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ LinearRegression     → RMSE: $29,414 | R²: 0.8045 | 4.0s


2026/03/02 21:34:42 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/02 21:34:45 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-fe02ac87-cb7b-4860-bcc0-26/tmpgz11_5k0/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/02 21:34:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ DecisionTree         → RMSE: $25,100 | R²: 0.8576 | 3.1s


2026/03/02 21:35:45 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/02 21:35:47 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-fe02ac87-cb7b-4860-bcc0-26/tmpktmkyhk4/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/02 21:35:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

2026/03/02 21:40:08 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/02 21:40:11 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-fe02ac87-cb7b-4860-bcc0-26/tmpt7tkh2rz/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/02 21:40:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ GBT                  → RMSE: $36,727 | R²: 0.6952 | 233.3s


2026/03/02 21:40:19 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/02 21:40:22 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-fe02ac87-cb7b-4860-bcc0-26/tmph4gde2n6/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/02 21:40:22 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ ElasticNet           → RMSE: $29,414 | R²: 0.8045 | 1.2s


model,rmse,r2,time_s
LinearRegression,29414.292966822013,0.8044967039379269,4.019715070724487
DecisionTree,25100.387926606705,0.857636674754714,3.1412570476531982
RandomForest,20549.66197256917,0.904578416078976,41.47567391395569
GBT,36726.822692811234,0.6952077127491725,233.3437535762787
ElasticNet,29414.290907965235,0.8044967313064768,1.2337219715118408


In [0]:
# ═══════════════════════════════════════════════════════════
# CELL 2: CROSSVALIDATOR + HYPERPARAMETER TUNING
# Rubric: "CrossValidator with proper parallelism" ✅
# ═══════════════════════════════════════════════════════════

import os
# Set Spark ML temp DFS path to a Unity Catalog volume for shared/serverless clusters
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/fema_claims_project/mlflow_tmp"

# OOM mitigation: Sample the training data for CV 
cv_train = train_p.sample(fraction=1.0, seed=42)

# Reduce grid size for RandomForestRegressor to avoid OOM
best_rf = RandomForestRegressor(featuresCol="features", labelCol=label)
# Reduced number of trees
rf_grid = ParamGridBuilder() \
    .addGrid(best_rf.numTrees, [50, 100]) \
    .addGrid(best_rf.maxDepth, [5, 10]) \
    .build()

# Reduce numFolds and parallelism for CV to mitigate OOM
rf_cv = CrossValidator(estimator=best_rf, estimatorParamMaps=rf_grid,
    evaluator=evaluator, numFolds=2, parallelism=2, seed=42)  # Reduced folds and parallelism

with mlflow.start_run(run_name="REG_RF_CrossValidator"):
    t0 = time.time()
    rf_cv_model = rf_cv.fit(cv_train)  # Fitting on sampled training data
    cv_time = time.time() - t0
    best_rmse = evaluator.evaluate(rf_cv_model.transform(val_p))
    mlflow.log_metric("best_rmse", best_rmse)
    mlflow.log_metric("cv_time_s", cv_time)
print(f"✅ Best RF (CV) → RMSE: ${best_rmse:,.0f} | Time: {cv_time:.1f}s")

# Save best model to Unity Catalog volume
rf_cv_model.bestModel.write().overwrite().save("/Volumes/workspace/default/fema_claims_project/best_regression")

✅ Best RF (CV) → RMSE: $20,550 | Time: 329.8s
